# **Analysis of Green House Gases Emitted by America's 3rd Largest City**

In an effort to stave off the environmental crisis that is predicted to occur later this century, many cities have begun to implement policies with the objective of reducing the ecological impact that urban places have on the environment due to their advanced industrialization. Thus, our goal is to analyze the energy program of one such city, Chicago, Illinois, to measure the success of their efforts as a whole, and locate any other trends regarding energy use in the city. 



**Imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
COLUMN_RENAME = {
    'Data Year': 'data_year',
    'ID': 'id',
    'Property Name': 'property_name',
    'Reporting Status': 'reporting_status',
    'Address': 'address',
    'ZIP Code': 'zip_code',
    'Chicago Energy Rating': 'chicago_energy_rating',
    'Exempt From Chicago Energy Rating': 'exempt_from_chicago_energy_rating',
    'Community Area': 'community_area',
    'Primary Property Type': 'primary_property_type',
    'Gross Floor Area - Buildings (sq ft)': 'gross_floor_area_buildings_sq_ft',
    'Year Built': 'year_built',
    '# of Buildings': 'number_of_buildings',
    'Water Use (kGal)': 'water_use_kgal',
    'ENERGY STAR Score': 'energy_star_score',
    'Electricity Use (kBtu)': 'electricity_use_kbtu',
    'Natural Gas Use (kBtu)': 'natural_gas_use_kbtu',
    'District Steam Use (kBtu)': 'district_steam_use_kbtu',
    'District Chilled Water Use (kBtu)': 'district_chilled_water_use_kbtu',
    'All Other Fuel Use (kBtu)': 'all_other_fuel_use_kbtu',
    'Site EUI (kBtu/sq ft)': 'site_eui_kbtu_sq_ft',
    'Source EUI (kBtu/sq ft)': 'source_eui_kbtu_sq_ft',
    'Weather Normalized Site EUI (kBtu/sq ft)': 'weather_normalized_site_eui_kbtu_sq_ft',
    'Weather Normalized Source EUI (kBtu/sq ft)': 'weather_normalized_source_eui_kbtu_sq_ft',
    'Total GHG Emissions (Metric Tons CO2e)': 'total_ghg_emissions_metric_tons_co2e',
    'GHG Intensity (kg CO2e/sq ft)': 'ghg_intensity_kg_co2e_sq_ft',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Location': 'location',
    'Row_ID': 'row_id',
}

# Read the committed snapshot (Chicago Energy Benchmarking, City of Chicago Data
# Portal) instead of the live API, so results are reproducible instead of
# depending on the dataset's current live state. Rename the portal export's
# human-readable headers to the API's snake_case field names this notebook uses.
df = pd.read_csv('Chicago_Energy_Benchmarking.csv')
df = df.rename(columns=COLUMN_RENAME)

# The portal export formats large numbers with thousands separators
# (e.g. "104,849"), which pandas reads as text. Strip the commas and
# coerce these columns back to numeric.
comma_formatted_columns = [
    'gross_floor_area_buildings_sq_ft', 'water_use_kgal', 'electricity_use_kbtu',
    'natural_gas_use_kbtu', 'district_steam_use_kbtu', 'district_chilled_water_use_kbtu',
    'all_other_fuel_use_kbtu', 'site_eui_kbtu_sq_ft', 'source_eui_kbtu_sq_ft',
    'weather_normalized_site_eui_kbtu_sq_ft', 'weather_normalized_source_eui_kbtu_sq_ft',
    'total_ghg_emissions_metric_tons_co2e',
]
for col in comma_formatted_columns:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')

### **Data Cleaning** 

**Initial Data Quality Check**

In [ ]:
print('Rows, columns:', df.shape)
print()
print('Missing values per column:')
print(df.isna().sum().sort_values(ascending=False))
print()
print('Duplicate (id, data_year) records:', df.duplicated(subset=['id', 'data_year']).sum())
print('Unique buildings (id):', df['id'].nunique(), 'across', df['data_year'].nunique(), 'reporting years')

In [54]:
north = [
    'Rogers Park',
    'Edgewater',
    'Lincoln Square',
    'Uptown',
    'North Center',
    'Roscoe Village',
    'Lakeview',
    'Lincoln Park',
    'Old Town',
    'Gold Coast',
    'River North',
    'Streeterville',
    'Near North Side',
    'Albany Park',
    'Lake View',
    'Dunning',
    'West Ridge',
    'Norwood Park',
    "O'Hare",
    'Montclare',
    'North Park',
    'Edison Park',
]

west = [
    'Forest Glen',
    'Jefferson Park',
    'Portage Park',
    'Irving Park',
    'Avondale',
    'Logan Square',
    'Bucktown',
    'Wicker Park',
    'Humboldt Park',
    'West Town',
    'West Loop',
    'Little Italy',
    'University Village',
    'Pilsen',
    'Belmont Cragin',
    'Hermosa'

]

south = [
    'Bridgeport',
    'Armour Square',
    'Bronzeville',
    'South Loop',
    'Hyde Park',
    'Kenwood',
    'South Shore',
    'Beverly',
    'Mckinley Park',
    'Near South Side'
]

In [ ]:
before = len(df)
df = df.dropna(subset=['community_area'])
df.reset_index(drop=True, inplace=True)
print(f'Dropped {before - len(df)} rows with no community_area')

communities_featured = df['community_area'].unique()
communities_featured

In [ ]:
df['community_area'] = df['community_area'].str.title()

In [57]:
hoods = df['community_area'].unique()
neighborhoods = [*north, *west, *south]
missing_hoods = []
for n in hoods:
  if n not in neighborhoods:
    missing_hoods.append(n)

In [58]:
def checkHoods(l1, l2, l3):
  hoods = df['community_area'].unique()
  
  neighborhoods = [*l1, *l2, *l3]

  missing_hoods = []
  
  for n in hoods:
    if n not in neighborhoods:
      missing_hoods.append(n)

  return missing_hoods


In [ ]:
df['community_area'] = df['community_area'].replace('Ohare', "O'Hare")
checkHoods(north, west, south)

In [ ]:
region_map = {}
region_map.update({hood: 'North' for hood in north})
region_map.update({hood: 'South' for hood in south})
region_map.update({hood: 'West' for hood in west})
region_map['Loop'] = 'Loop'

df['region'] = df['community_area'].map(region_map).fillna('Outside Chicago, In Bedford Park')

In [61]:
df.head()

,data_year,id,property_name,reporting_status,address,zip_code,chicago_energy_rating,exempt_from_chicago_energy_rating,community_area,primary_property_type,...,source_eui_kbtu_sq_ft,weather_normalized_site_eui_kbtu_sq_ft,weather_normalized_source_eui_kbtu_sq_ft,total_ghg_emissions_metric_tons_co2e,ghg_intensity_kg_co2e_sq_ft,latitude,longitude,location,row_id,region
0,2020,252064,Mansueto Library,Submitted Data,1100 E 57th St,60637,2.0,False,Hyde Park,Library,...,323.6,246.0,329.9,1160.9,18.1,41.792213,-87.599950,"(41.79221307, -87.59994981)",2020-252064,South
1,2020,232458,Harper Square Cooperative,Submitted Data,4800 - 4850 S Lake Park Ave,60615,1.0,False,Kenwood,Multifamily Housing,...,146.0,100.3,150.7,4871.7,7.8,41.807475,-87.591264,"(41.80747487, -87.59126397)",2020-232458,South
2,2020,254616,Former Coyne College,Submitted Data,330 N Green St,60607,2.0,False,Near West Side,Office,...,148.3,56.7,151.8,4581.4,8.2,41.873335,-87.651021,"(41.873335, -87.65102071)",2020-254616,"Outside Chicago, In Bedford Park"
3,2020,103812,400 W Superior St,Submitted Data,400 W Superior St,60654,3.0,False,Near North Side,Office,...,151.8,63.0,154.8,1092.1,8.4,41.895752,-87.638901,"(41.89575232, -87.638901)",2020-103812,North
4,2020,254073,Blue Moon Lofts,Submitted Data,215 N. Aberdeen St.,60607,4.0,False,Near West Side,Multifamily Housing,...,64.9,29.6,64.3,295.8,3.6,41.874295,-87.650175,"(41.87429514, -87.65017516)",2020-254073,"Outside Chicago, In Bedford Park"


## **GHG Emissions Trend, 2014-2023**

As federal greenhouse-gas regulation has been rolled back through 2025-2026 (the EPA repealed its power-plant GHG rule in September 2026), city-level building benchmarking ordinances like Chicago's are increasingly the primary lever left for cutting building emissions. With three more years of reporting now available (2021-2023) than the original version of this project had, this section asks two questions the 2020 snapshot couldn't:

1. **Has citywide GHG intensity actually declined, or does it just look that way because the mix of reporting buildings changed?** Chicago's ordinance phased in by building size, so the *early* reporting pool (243 buildings in 2014) is a very different population than the *later* one (3,438 in 2023).
2. **Among buildings tracked over multiple years, are individual buildings actually reducing their emissions intensity?** This is the stronger, more honest version of the question: the same building, year over year, rather than a citywide average that a changing population could distort.

**Question 1: Citywide trend vs. a fixed panel of repeat reporters**

In [ ]:
yearly_all = df.groupby('data_year').agg(
    n_reporting=('id', 'count'),
    median_ghg_intensity=('ghg_intensity_kg_co2e_sq_ft', 'median'),
).reset_index()
yearly_all

In [ ]:
# Restrict to buildings with a real multi-year track record, so the "trend"
# isn't just a symptom of which buildings happen to report in a given year.
has_intensity = df.dropna(subset=['ghg_intensity_kg_co2e_sq_ft'])
years_reported = has_intensity.groupby('id')['data_year'].nunique()
panel_ids = years_reported[years_reported >= 4].index

yearly_panel = (
    has_intensity[has_intensity['id'].isin(panel_ids)]
    .groupby('data_year')['ghg_intensity_kg_co2e_sq_ft']
    .median()
    .reset_index(name='median_ghg_intensity')
)

print(f'{len(panel_ids)} of {df["id"].nunique()} unique buildings have >=4 years '
      'of GHG intensity data and make up the repeat-reporter panel')

In [ ]:
CITYWIDE_COLOR = '#0072B2'  # blue
PANEL_COLOR = '#D55E00'     # vermillion

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(yearly_all['data_year'], yearly_all['median_ghg_intensity'],
        color=CITYWIDE_COLOR, linewidth=2, marker='o', markersize=8,
        label='All reporting buildings that year')
ax.plot(yearly_panel['data_year'], yearly_panel['median_ghg_intensity'],
        color=PANEL_COLOR, linewidth=2, marker='o', markersize=8,
        label='Repeat reporters only (\u2265 4 years of data)')

ax.set_xlabel('Data year')
ax.set_ylabel('Median GHG intensity (kg CO\u2082e / sq ft)')
ax.set_title('The decline holds up even when the reporting population is held fixed')
ax.grid(axis='y', color='#dddddd', linewidth=0.8, zorder=0)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The two lines are nearly on top of each other. Restricting to the 2,933 buildings with a real multi-year history (76% of all unique buildings) barely changes the median GHG intensity in any year, compared to using every building that happened to report that year. That's evidence *against* the composition-change hypothesis: the citywide decline from ~13 to ~6.2 kg CO2e/sq ft isn't primarily an artifact of which buildings are reporting when — it holds up under a same-population check.

That's still a same-year comparison of two overlapping groups, not a true balanced panel (a building only contributes to a given year's panel median if it actually reported that year). Question 2 goes further and looks at each building's own trend over time.

**Question 2: Are individual buildings actually improving?**

In [ ]:
panel = has_intensity[has_intensity['id'].isin(panel_ids)]

def year_over_year_slope(building_years):
    x = building_years['data_year'].to_numpy(dtype=float)
    y = building_years['ghg_intensity_kg_co2e_sq_ft'].to_numpy(dtype=float)
    return np.polyfit(x, y, 1)[0]

slopes = panel.groupby('id').apply(year_over_year_slope, include_groups=False)
slopes.name = 'ghg_intensity_slope'

FLAT_THRESHOLD = 0.05  # kg CO2e/sq ft per year; smaller than this counts as "flat"
improved = (slopes < -FLAT_THRESHOLD).sum()
worsened = (slopes > FLAT_THRESHOLD).sum()
flat = len(slopes) - improved - worsened

print(f'Improved (falling GHG intensity): {improved} ({improved / len(slopes):.1%})')
print(f'Worsened (rising GHG intensity):  {worsened} ({worsened / len(slopes):.1%})')
print(f'Roughly flat:                     {flat} ({flat / len(slopes):.1%})')
print(f'Median per-building slope: {slopes.median():.3f} kg CO2e/sq ft per year')

In [ ]:
IMPROVED_COLOR = '#0072B2'  # blue
WORSENED_COLOR = '#D55E00'  # vermillion
FLAT_COLOR = '#999999'      # gray

fig, ax = plt.subplots(figsize=(9, 5))
clipped = slopes.clip(-3, 3)  # a handful of extreme outliers would otherwise flatten the histogram
bins = np.linspace(-3, 3, 61)
colors = np.where(clipped < -FLAT_THRESHOLD, IMPROVED_COLOR,
          np.where(clipped > FLAT_THRESHOLD, WORSENED_COLOR, FLAT_COLOR))

for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (clipped >= lo) & (clipped < hi)
    if mask.sum() == 0:
        continue
    mid = (lo + hi) / 2
    color = IMPROVED_COLOR if mid < -FLAT_THRESHOLD else WORSENED_COLOR if mid > FLAT_THRESHOLD else FLAT_COLOR
    ax.bar(mid, mask.sum(), width=(hi - lo) * 0.9, color=color)

ax.axvline(0, color='#444444', linewidth=1, linestyle='--')
ax.set_xlabel('Per-building GHG intensity trend (kg CO\u2082e / sq ft per year, clipped to \u00b13)')
ax.set_ylabel('Number of buildings')
ax.set_title('Most repeat-reporting buildings are independently trending down')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='#dddddd', linewidth=0.8, zorder=0)
plt.tight_layout()
plt.show()

Both checks point the same direction: the citywide decline in GHG intensity is not just a reshuffling of which buildings report, and it isn't driven by a handful of large improvers either — the majority of individual buildings that have been tracked for at least 4 years are independently trending down. That's a meaningfully stronger claim than the original project's single-year snapshot could support.